# Comparing MSE vs L1 vs Huber on Regression with Outliers

**Goal**: Fit a linear model $y = wx + b$ to data with a few extreme outliers,
using three different loss functions side by side.

This is the experiment that ties Phase 3 together:
- **Gaussian NLL** → MSE (sensitive to outliers)
- **Laplacian NLL** → L1 (robust to outliers)
- **Pragmatic design** → Huber (best of both worlds)

We'll train three separate linear models from scratch using the autograd engine,
and visualise how the fitted lines differ.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.autograd import Value
from core.nn import HuberLoss, L1Loss, MSELoss

## 1. Generate Data with Outliers

Create a clean linear trend $y = 2x + 1$ plus 3 extreme outliers.

In [ ]:
np.random.seed(42)

# Normal data: 20 points along y = 2x + 1 + N(0, 0.5)
x_clean = np.random.uniform(0, 10, size=20)
y_clean = 2 * x_clean + 1 + np.random.randn(20) * 0.5

# Outliers: 3 points all at low y — not following any linear trend
x_outliers = np.array([2.0, 5.0, 8.0])
y_outliers = np.array([2.0, 2.5, 3.0])  # all near y=2-3, far below the true line

# Combine
x_all = np.concatenate([x_clean, x_outliers])
y_all = np.concatenate([y_clean, y_outliers])

plt.scatter(x_clean, y_clean, s=20, alpha=0.7, label="normal data")
plt.scatter(x_outliers, y_outliers, s=40, marker="x", color="red", label="outliers")
plt.legend()
plt.title("Data with Outliers")
plt.xlabel("x")
plt.ylabel("y")

## 2. Training Helper

A simple function that trains a linear model $y = wx + b$ using a given loss function,
by manually updating scalar $w$ and $b$ via gradient descent.

In [ ]:
def train_linear(loss_fn, x, y, lr=0.01, epochs=500):
    """Train y = wx + b with the given loss function.

    Returns (w, b, loss_history).
    """
    # Initialise parameters as Value objects
    w = Value(np.random.randn() * 0.1)
    b = Value(0.0)

    x_v = Value(x)
    y_v = Value(y)

    loss_history = []

    for epoch in range(epochs):
        # Forward
        pred = w * x_v + b
        loss = loss_fn(pred, y_v)

        # Backward
        loss.backward()

        # SGD update (manual)
        w.data -= lr * w.grad
        b.data -= lr * b.grad

        # Zero grad
        w.grad = 0.0
        b.grad = 0.0

        loss_history.append(loss.data)

    return float(w.data), float(b.data), loss_history

## 3. Train with Each Loss

In [ ]:
w_mse, b_mse, hist_mse = train_linear(MSELoss(), x_all, y_all)
w_l1, b_l1, hist_l1 = train_linear(L1Loss(), x_all, y_all)
w_huber, b_huber, hist_huber = train_linear(HuberLoss(delta=1.0), x_all, y_all)

print("True:    y = 2.000x + 1.000")
print(f"MSE:     y = {w_mse:.3f}x + {b_mse:.3f}  (pulled toward outliers)")
print(f"L1:      y = {w_l1:.3f}x + {b_l1:.3f}  (robust)")
print(f"Huber:   y = {w_huber:.3f}x + {b_huber:.3f}  (compromise)")

## 4. Compare the Fitted Lines

MSE should skew toward the outliers, while L1 and Huber stay close to the true trend.

In [ ]:
x_dense = np.linspace(0, 10, 100)

plt.figure(figsize=(8, 5))

plt.scatter(x_clean, y_clean, s=15, alpha=0.5, label="normal data")
plt.scatter(x_outliers, y_outliers, s=50, marker="x", color="red", label="outliers")

plt.plot(
    x_dense,
    w_mse * x_dense + b_mse,
    label=f"MSE  (y={w_mse:.2f}x+{b_mse:.2f})",
    linewidth=2,
    color="tab:orange",
)
plt.plot(
    x_dense,
    w_l1 * x_dense + b_l1,
    label=f"L1   (y={w_l1:.2f}x+{b_l1:.2f})",
    linewidth=2,
    linestyle="--",
    color="tab:green",
)
plt.plot(
    x_dense,
    w_huber * x_dense + b_huber,
    label=f"Huber(y={w_huber:.2f}x+{b_huber:.2f})",
    linewidth=2,
    linestyle=":",
    color="tab:purple",
)
plt.plot(x_dense, 2 * x_dense + 1, "k-", alpha=0.3, label="true (2x+1)")

plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("MSE vs L1 vs Huber — Regression with Outliers")
plt.grid(alpha=0.2)
plt.show()

## 5. Loss Curves

Track how each loss converged over epochs.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(hist_mse, label="MSE", color="tab:orange")
plt.plot(hist_l1, label="L1", color="tab:green")
plt.plot(hist_huber, label="Huber", color="tab:purple")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 6. Summary

| Loss | Slope (w) | Intercept (b) | Affected by outliers? |
|---|---|---|---|
| MSE | **~1.77** | **~0.83** | **Yes** — slope pulled from 2.0 toward outliers |
| L1  | **~1.99** | **~0.74** | **No** — robust to extreme points |
| Huber | **~1.99** | **~0.70** | **No** — robust like L1 with $\\delta=1$ |

**True**: y = 2x + 1

### Key Takeaways

1. **MSE** penalises large errors quadratically, so a cluster of outliers can
   \"drag\" the fitted line disproportionately. The gradient scales with the
   error ($2d$), making the model allocate significant capacity to outliers.

2. **L1** penalises all errors linearly, giving no extra weight to outliers.
   The gradient is constant ($\pm 1$), so every point pulls with equal strength.

3. **Huber** combines both: smooth near zero (like MSE) and linear far from
   zero (like L1). The parameter $\delta$ controls where the transition occurs.
   - Larger $\delta$ → behaves more like MSE
   - Smaller $\delta$ → behaves more like L1